# YOLOX stage 3b: 4-class football detector with drone and rig hard negatives

Same recipe, seed and hyper-parameters as the stage-1 notebook (`yolox_finetune_colab.ipynb`), from COCO weights, with **one change: the dataset**. `finetune_v3_rig.zip` is stage 1's data plus 140 box-less images: 75 (15 patches x 5) of the hanging camera rig that the ball detector mistakes for the ball, and 65 (13 patches x 5) of the same rig that the detector tracks as a *person* (`docs/finetune_4class.md`, sections "Drone hard negatives" and "Rig as a person"). Everything else is held fixed so any difference is attributable to the negatives (one seed, so run-to-run noise is not separated).

**Nothing is lost if Colab disconnects.** YOLOX is told to write straight into a Google Drive folder (`output_dir`), so
(checked in the YOLOX source at the pinned commit):

| File in `MyDrive/football_cv/yolox_runs/<exp>/` | Written | Use |
|---|---|---|
| `latest_ckpt.pth` | after **every epoch** | resume point (model, EMA, optimiser, epoch) |
| `best_ckpt.pth` | at each evaluation, when COCO AP50-95 improves | the model to keep |
| `last_epoch_ckpt.pth` | at each evaluation | last evaluated weights |
| `train_log.txt`, `tensorboard/` | continuously | history |
| `football4_rig_exp.py`, `run_config.json`, `dataset_manifest.json`, `sessions.log` | at start of every session | the config the run used |

If the session dies, reconnect and **Run all**: the training cell finds `latest_ckpt.pth` and resumes (`--resume`).
Drive syncs in the background, so the last few seconds before a crash can be missing; at most one epoch is repeated.

- **Validation = the three audited evaluation windows** (153 frames). Nothing from them is in the training set.
- COCO AP printed here is a **training signal only**. The project score (HOTA, role accuracy, ball hits) comes from `scripts/eval_tracking.py` after the checkpoint is wired into the pipeline.
- Hyper-parameters are starting points, **not tuned or measured**. Pass thresholds are in `docs/finetune_4class.md`.
- **Pass criteria were written before this run** (`docs/finetune_4class.md`): the drone and real-ball checks come from `scripts/eval_drone_check.py` run locally on the downloaded checkpoint; the notebook's COCO AP does not decide them.
- **Not executed yet**: expect to adjust package versions if Colab's Python/torch moved.

Before running: upload `finetune_v3_rig.zip` (the build script's output folder, zipped) to `MyDrive/football_cv/`.

In [ ]:
import subprocess, torch, os, pathlib, json, datetime, hashlib, shutil
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/football_cv'
DATASET_ZIP = f'{DRIVE}/finetune_v3_rig.zip'
EXP_NAME = 'yolox_m_football4_rig'
RUN_ROOT = f'{DRIVE}/yolox_runs'          # YOLOX writes its checkpoints straight into this Drive folder
RUN_DIR = f'{RUN_ROOT}/{EXP_NAME}'
SEED = 0
BATCH = 8                                  # lower this first if the T4 runs out of memory
os.makedirs(RUN_DIR, exist_ok=True)
print('checkpoints and logs go to', RUN_DIR)
print('already there:', sorted(os.listdir(RUN_DIR)))

## 1. YOLOX at a pinned commit (Apache-2.0)

In [ ]:
YOLOX_COMMIT = '6ddff4824372906469a7fae2dc3206c7aa4bbaee'
if not os.path.isdir('/content/YOLOX'):
    !git clone https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX
%cd /content/YOLOX
!git checkout {YOLOX_COMMIT}
# Do NOT run `pip install -r requirements.txt` or a plain `pip install -e .`: both pull the pinned onnx-simplifier==0.4.10
# (ONNX export only, not needed here), whose source build fails on Colab's Python ("metadata-generation-failed").
!pip install -q loguru tqdm thop ninja tabulate psutil tensorboard pycocotools opencv-python
# No `pip install -e .`: its C++ fast-COCO-eval build fails on Colab's Python 3.13 and is optional (YOLOX falls back to
# pycocotools' COCOeval, same numbers, slower). The source checkout on PYTHONPATH is enough.
import sys
os.environ['PYTHONPATH'] = '/content/YOLOX'
if '/content/YOLOX' not in sys.path:
    sys.path.insert(0, '/content/YOLOX')
# Colab's recent torch loads checkpoints with weights_only=True, which rejects some objects YOLOX saves (e.g. numpy scalars).
# The files loaded here are the official COCO checkpoint (SHA-256 printed below) and this notebook's own Drive checkpoints.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
import yolox
print('yolox', yolox.__version__, yolox.__file__)

In [ ]:
# COCO-pretrained YOLOX-m (URL from the YOLOX README benchmark table); its SHA-256 is printed so the run is traceable.
if not os.path.exists('/content/YOLOX/yolox_m.pth'):
    !wget -q -O /content/YOLOX/yolox_m.pth https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_m.pth
!sha256sum /content/YOLOX/yolox_m.pth && ls -l /content/YOLOX/yolox_m.pth

## 2. Dataset
YOLOX's `COCODataset` reads `<data_dir>/annotations/<json>` and images from `<data_dir>/train2017` and `<data_dir>/val2017`. The zip is unpacked to local disk (fast); it is re-unpacked in every new session.

In [ ]:
import zipfile
DATA = pathlib.Path('/content/YOLOX/datasets/football4_rig')
if not (DATA / 'manifest.json').exists():
    DATA.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(DATA)
if not (DATA / 'annotations').exists():                 # zip made from the parent folder
    DATA = next(p for p in DATA.iterdir() if (p / 'annotations').exists())
man = json.loads((DATA / 'manifest.json').read_text())
print('classes', man['classes'])
print('train', man['train'])
print('val  ', man['val'])
for split in ('train', 'val'):
    coco = json.loads((DATA / 'annotations' / f'instances_{split}2017.json').read_text())
    n_img = len(list((DATA / f'{split}2017').glob('*.jpg')) + list((DATA / f'{split}2017').glob('*.png')))
    assert len(coco['images']) == n_img == man[split]['images'], (split, len(coco['images']), n_img)
assert man['classes'] == ['ball', 'goalkeeper', 'player', 'referee']
assert man['train']['boxes'].get('referee', 0) > 0 and man['train']['boxes'].get('ball', 0) > 0, 'no referee/ball in training data'
assert man['negatives']['images_added'] == 140, man.get('negatives')
assert man['train']['boxes'] == {'referee': 2589, 'player': 23948, 'goalkeeper': 977, 'ball': 1243}, 'box counts differ from stage 1'
shutil.copy2(DATA / 'manifest.json', f'{RUN_DIR}/dataset_manifest.json')
print('dataset OK at', DATA)

## 3. Experiment file and run config (saved to Drive)

Inherits `yolox.exp.Exp` and overrides only what changes. Input size is wide because the ball is a few pixels at 1280 px. Mixup is off for the same reason. `output_dir` points at Drive. `save_history_ckpt = False` stops YOLOX writing a separate full checkpoint at every evaluation (about 30 extra files of a few hundred MB each); `latest_ckpt.pth` and `best_ckpt.pth` are what matter.

In [ ]:
exp_src = f'''
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super().__init__()
        self.depth, self.width = 0.67, 0.75                    # YOLOX-m
        self.exp_name = '{EXP_NAME}'
        self.output_dir = r'{RUN_ROOT}'                         # Drive: checkpoints survive a dead session
        self.save_history_ckpt = False
        self.num_classes = 4                                    # ball, goalkeeper, player, referee
        self.data_dir = r'{DATA}'
        self.train_ann = 'instances_train2017.json'
        self.val_ann = 'instances_val2017.json'
        self.input_size = (704, 1280)                           # (h, w), multiples of 32
        self.test_size = (704, 1280)
        self.multiscale_range = 3
        self.enable_mixup = False
        self.mosaic_prob = 0.8
        self.max_epoch = 60
        self.warmup_epochs = 3
        self.no_aug_epochs = 10
        self.eval_interval = 2                                  # best_ckpt can only improve at an evaluation
        self.print_interval = 20
        self.data_num_workers = 2                               # Colab reports 2 CPUs as the sensible maximum
        self.seed = {SEED}
'''
pathlib.Path('/content/YOLOX/exps/football4_rig.py').write_text(exp_src)
pathlib.Path(f'{RUN_DIR}/football4_rig_exp.py').write_text(exp_src)

now = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
env = {'utc': now, 'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0), 'batch': BATCH}
if not os.path.exists(f'{RUN_DIR}/run_config.json'):
    pathlib.Path(f'{RUN_DIR}/run_config.json').write_text(json.dumps(
        {'exp_name': EXP_NAME, 'yolox_commit': YOLOX_COMMIT, 'seed': SEED, 'pretrained': 'yolox_m.pth (COCO)',
         'first_session': env, 'dataset_zip': DATASET_ZIP, 'dataset_counts': {k: man[k] for k in ('train', 'val')},
         'exp_file': 'football4_rig_exp.py'}, indent=2))
with open(f'{RUN_DIR}/sessions.log', 'a') as f:
    f.write(json.dumps(env) + '\n')
print(exp_src)

## 4. Train (auto-resumes)

Fresh run: fine-tunes from the COCO checkpoint (`-c yolox_m.pth`; the 80-class head does not match the 4-class head, so YOLOX skips those weights and leaves the new head freshly initialised). If `latest_ckpt.pth` exists on Drive it runs with `--resume` instead, which reloads model, EMA, optimiser and epoch from that file. The log is appended to `train_log.txt` on Drive.

In [ ]:
latest = pathlib.Path(RUN_DIR) / 'latest_ckpt.pth'
if latest.exists():
    mode = '--resume'
    print('RESUMING from', latest, f'({latest.stat().st_size/1e6:.0f} MB)')
else:
    mode = '-c yolox_m.pth'
    print('fresh run from COCO weights')
# GPU utilisation and memory every 15 s to Drive, so "is the GPU busy?" has a measured answer (gpu_log.csv).
# (YOLOX's own "gpu mem" figure is not usage when -o is on; -o is left off here.)
mon = subprocess.Popen(['nvidia-smi', '--query-gpu=timestamp,utilization.gpu,memory.used,memory.total',
                        '--format=csv,noheader', '-l', '15'], stdout=open(f'{RUN_DIR}/gpu_log.csv', 'a'))
try:
    !cd /content/YOLOX && python tools/train.py -f exps/football4_rig.py -expn {EXP_NAME} -d 1 -b {BATCH} --fp16 {mode} 2>&1 | tee -a {RUN_DIR}/train_log.txt
finally:
    mon.terminate()

## 5. What is saved on Drive (run this first after a reconnect)

In [ ]:
for p in sorted(pathlib.Path(RUN_DIR).glob('*')):
    if p.is_file():
        print(f'{p.name:28s} {p.stat().st_size/1e6:9.1f} MB   {datetime.datetime.fromtimestamp(p.stat().st_mtime):%Y-%m-%d %H:%M}')

## 6. Evaluate the best checkpoint on the held-out windows
COCO AP on the three audited windows. Compare the **referee** and **ball** rows with the thresholds in `docs/finetune_4class.md`; the project metrics come later from `scripts/eval_tracking.py`.

In [ ]:
CKPT = f'{RUN_DIR}/best_ckpt.pth'
assert os.path.exists(CKPT), f'no best_ckpt.pth in {RUN_DIR}: run the training cell (section 4) first. It appears after the first evaluation (every 2 epochs). Files there now: {sorted(os.listdir(RUN_DIR))}'
!cd /content/YOLOX && python tools/eval.py -f exps/football4_rig.py -expn {EXP_NAME} -c {CKPT} -b 8 -d 1 --conf 0.01 --fp16 --fuse 2>&1 | tee {RUN_DIR}/eval_log.txt | tail -n 50

## 7. Freeze a final copy
Copies the best checkpoint, config, logs and manifest into a timestamped `final_*` folder with a SHA-256, so a later resume or re-run cannot overwrite what you decided to keep.

In [ ]:
final = pathlib.Path(RUN_DIR) / f'final_{datetime.datetime.now(datetime.timezone.utc):%Y%m%d_%H%M}'
final.mkdir()
for name in ('best_ckpt.pth', 'football4_rig_exp.py', 'run_config.json', 'dataset_manifest.json', 'train_log.txt', 'eval_log.txt', 'sessions.log'):
    if os.path.exists(f'{RUN_DIR}/{name}'):
        shutil.copy2(f'{RUN_DIR}/{name}', final)
sha = hashlib.sha256(open(final / 'best_ckpt.pth', 'rb').read()).hexdigest()
(final / 'SHA256').write_text(f'{sha}  best_ckpt.pth\nyolox_commit {YOLOX_COMMIT}\nseed {SEED}\n')
os.sync()
print('saved to', final, '\nsha256', sha)

## Next (on the local machine)
Copy `best_ckpt.pth` to `models/football_yolox/`, add the YOLOX backend to `src/football_cv/detection/detector.py` behind the same `role` output, then score with `scripts/eval_tracking.py` against `tracks_v0.parquet` and the 4-class baseline. Weights stay out of git.